# Inference Evaluation Notebook
This notebook provides an **industry-style evaluation workflow** for inference runs created from either:
- `notebook/inference.ipynb`
- Chainlit UI in `app.py`

## Primary Data Source
This notebook now treats **Phoenix serve** as the primary evaluation store.
Phoenix is the system of record for observability data such as traces, spans, token usage, inputs, outputs, and retrieval telemetry.

## What This Notebook Extracts From Phoenix
The notebook can build multiple dataframes at different granularities:
- **Trace-level dataframe**: one row per trace with token totals and timing
- **Span-level dataframe**: one row per span for low-level debugging
- **Turn-level dataframe**: one row per user-facing turn span
- **LLM dataframe**: one row per LLM-related span with model and token fields
- **Retrieval dataframe**: one row per retrieved document emitted by Phoenix retriever spans

## Important Caveat
Phoenix can only show data that was emitted during the original inference run.
If a field was never sent to Phoenix during inference, it cannot be reconstructed from Phoenix later.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "app.py").exists() and (project_root.parent / "app.py").exists():
    project_root = project_root.parent
    print('project_root: ', project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

project_root:  C:\Program Files\Studying\coding\RAG_project


In [2]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd
from phoenix.client import Client

from src.rag.core.observability import (
    DEFAULT_PHOENIX_ENDPOINT,
    DEFAULT_PHOENIX_PROJECT_NAME,
    PHOENIX_INSTALL_COMMAND,
 )


PROJECT_ROOT = project_root
PHOENIX_BASE_URL = DEFAULT_PHOENIX_ENDPOINT
PHOENIX_PROJECT_NAME = DEFAULT_PHOENIX_PROJECT_NAME


@dataclass(frozen=True)
class EvaluationConfig:
    """Configuration object for Phoenix-backed evaluation.

    Purpose:
    - Keep all notebook evaluation settings in one typed object.
    - Make Phoenix endpoint, project name, and query limits explicit and reusable.

    Output:
    - Immutable configuration record used by helper functions below.
    """

    project_root: Path
    phoenix_base_url: str
    phoenix_project_name: str
    phoenix_span_limit: int = 5000
    phoenix_trace_limit: int = 1000


EVAL_CONFIG = EvaluationConfig(
    project_root=PROJECT_ROOT,
    phoenix_base_url=PHOENIX_BASE_URL,
    phoenix_project_name=PHOENIX_PROJECT_NAME,
)


def _safe_json_text(value: Any) -> str:
    """Serialize complex Python objects into stable JSON text for export.

    Purpose:
    - Preserve nested Phoenix attributes in CSV-friendly string form.

    Output:
    - JSON string when serialization succeeds.
    - Fallback string representation when JSON encoding is not possible.
    """
    try:
        return json.dumps(value, ensure_ascii=False, default=str)
    except Exception:
        return str(value)


def _extract_interface(metadata_value: Any) -> str | None:
    """Extract the interface name from Phoenix metadata.

    Purpose:
    - Phoenix metadata is often stored as a nested dictionary.
    - This helper normalizes the `interface` field for reporting.

    Output:
    - Interface label such as `notebook` or `chainlit`.
    - None when no interface information exists.
    """
    if isinstance(metadata_value, dict):
        value = metadata_value.get("interface")
        return str(value) if value is not None else None
    return None


def create_phoenix_client(config: EvaluationConfig) -> Client:
    """Create a Phoenix API client for reading evaluation data from Phoenix serve.

    Purpose:
    - Provide one shared construction point for Phoenix API access.

    Output:
    - Configured `phoenix.client.Client` instance.
    """
    return Client(base_url=config.phoenix_base_url)


def fetch_trace_catalog_dataframe(
    client: Client,
    project_name: str,
    limit: int = 1000,
    session_id: str | None = None,
 ) -> pd.DataFrame:
    """Fetch trace-level summary rows from Phoenix.

    Purpose:
    - Build a compact evaluation view with one row per trace.
    - Expose token totals and timing without requiring manual span inspection.

    Output:
    - DataFrame with trace id, timing, token counts, and optional session id filter applied.
    """
    traces = client.traces.get_traces(
        project_identifier=project_name,
        include_spans=False,
        session_id=session_id,
        limit=limit,
    )

    rows: list[dict[str, Any]] = []
    for trace in traces:
        start_time = pd.to_datetime(trace.get("start_time"), utc=True, errors="coerce")
        end_time = pd.to_datetime(trace.get("end_time"), utc=True, errors="coerce")
        latency_ms = None
        if pd.notna(start_time) and pd.notna(end_time):
            latency_ms = (end_time - start_time).total_seconds() * 1000.0

        rows.append(
            {
                "trace_row_id": trace.get("id"),
                "trace_id": trace.get("trace_id"),
                "project_id": trace.get("project_id"),
                "start_time": start_time,
                "end_time": end_time,
                "latency_ms": latency_ms,
                "token_count_prompt": trace.get("token_count_prompt"),
                "token_count_completion": trace.get("token_count_completion"),
                "token_count_total": trace.get("token_count_total"),
            }
        )

    trace_df = pd.DataFrame(rows)
    if not trace_df.empty:
        trace_df = trace_df.sort_values(by=["start_time"], ascending=False, na_position="last")
    return trace_df.reset_index(drop=True)


def fetch_spans_dataframe(client: Client, project_name: str, limit: int = 5000) -> pd.DataFrame:
    """Fetch raw span data from Phoenix for one project.

    Purpose:
    - Retrieve the detailed observability dataset used to derive all other evaluation tables.

    Output:
    - DataFrame with one row per span and Phoenix-provided attributes as columns.
    """
    spans_df = client.spans.get_spans_dataframe(
        project_name=project_name,
        limit=limit,
    )
    if spans_df.empty:
        return spans_df
    return spans_df.sort_values(by=["start_time"], ascending=False, na_position="last").reset_index(drop=True)


def fetch_available_session_ids(spans_df: pd.DataFrame) -> list[str]:
    """List distinct Phoenix session ids available in the current spans dataset.

    Purpose:
    - Help notebook users choose a valid session id instead of guessing one manually.

    Output:
    - Sorted list of unique session ids present in `attributes.session.id`.
    """
    session_column = "attributes.session.id"
    if spans_df.empty or session_column not in spans_df.columns:
        return []
    return sorted({str(value) for value in spans_df[session_column].dropna().tolist()})


def filter_spans_by_session(spans_df: pd.DataFrame, session_id: str) -> pd.DataFrame:
    """Filter Phoenix spans down to one saved chat session.

    Purpose:
    - Phoenix stores many sessions in one project.
    - This helper isolates the rows related to one `attributes.session.id`.

    Output:
    - DataFrame containing only spans that belong to the requested session id.
    """
    if spans_df.empty or "attributes.session.id" not in spans_df.columns:
        return spans_df.iloc[0:0].copy()

    filtered = spans_df[spans_df["attributes.session.id"] == session_id].copy()
    return filtered.reset_index(drop=True)


def build_span_overview_dataframe(spans_df: pd.DataFrame) -> pd.DataFrame:
    """Create a clean span-level overview for human inspection.

    Purpose:
    - Reduce the very wide Phoenix span table to the most important operational fields.

    Output:
    - DataFrame with one row per span and a curated subset of columns.
    """
    if spans_df.empty:
        return pd.DataFrame()

    overview_df = spans_df.copy()
    overview_df["interface"] = overview_df.get("attributes.metadata", pd.Series([None] * len(overview_df))).apply(_extract_interface)

    selected_columns = [
        "name",
        "span_kind",
        "status_code",
        "start_time",
        "end_time",
        "context.trace_id",
        "context.span_id",
        "parent_id",
        "attributes.session.id",
        "interface",
        "attributes.llm.model_name",
        "attributes.llm.token_count.prompt",
        "attributes.llm.token_count.completion",
        "attributes.llm.token_count.total",
        "attributes.input.value",
        "attributes.output.value",
    ]
    available_columns = [col for col in selected_columns if col in overview_df.columns]
    return overview_df[available_columns].copy()


def build_turn_dataframe_from_phoenix(spans_df: pd.DataFrame) -> pd.DataFrame:
    """Create a turn-level dataframe from Phoenix spans.

    Purpose:
    - Represent one user-facing turn per row using the turn spans emitted by the app or notebook.
    - Support both Chainlit (`rag.chat_turn`) and notebook (`CondensePlusContextChatEngine.chat`) flows.

    Output:
    - DataFrame with one row per detected turn span.
    """
    if spans_df.empty:
        return pd.DataFrame()

    turn_names = {"rag.chat_turn", "CondensePlusContextChatEngine.chat"}
    turn_df = spans_df[spans_df["name"].isin(turn_names)].copy()
    if turn_df.empty:
        return turn_df

    turn_df["interface"] = turn_df.get("attributes.metadata", pd.Series([None] * len(turn_df))).apply(_extract_interface)
    selected_columns = [
        "name",
        "start_time",
        "end_time",
        "status_code",
        "context.trace_id",
        "context.span_id",
        "attributes.session.id",
        "interface",
        "attributes.input.value",
        "attributes.output.value",
        "attributes.llm.token_count.prompt",
        "attributes.llm.token_count.completion",
        "attributes.llm.token_count.total",
    ]
    available_columns = [col for col in selected_columns if col in turn_df.columns]
    return turn_df[available_columns].sort_values(by=["start_time"], ascending=True, na_position="last").reset_index(drop=True)


def build_llm_dataframe_from_phoenix(spans_df: pd.DataFrame) -> pd.DataFrame:
    """Create an LLM-call dataframe from Phoenix spans.

    Purpose:
    - Isolate spans that contain model-level metadata or token accounting.
    - Make model usage and token analysis easy to export and audit.

    Output:
    - DataFrame with one row per LLM-related span.
    """
    if spans_df.empty:
        return pd.DataFrame()

    llm_mask = spans_df.get("attributes.llm.model_name", pd.Series([None] * len(spans_df))).notna()
    llm_mask = llm_mask | spans_df.get("attributes.llm.token_count.total", pd.Series([None] * len(spans_df))).notna()
    llm_df = spans_df[llm_mask].copy()
    if llm_df.empty:
        return llm_df

    llm_df["interface"] = llm_df.get("attributes.metadata", pd.Series([None] * len(llm_df))).apply(_extract_interface)
    selected_columns = [
        "name",
        "start_time",
        "end_time",
        "status_code",
        "context.trace_id",
        "context.span_id",
        "attributes.session.id",
        "interface",
        "attributes.llm.model_name",
        "attributes.llm.input_messages",
        "attributes.llm.output_messages",
        "attributes.llm.invocation_parameters",
        "attributes.llm.token_count.prompt",
        "attributes.llm.token_count.completion",
        "attributes.llm.token_count.total",
        "attributes.output.value",
    ]
    available_columns = [col for col in selected_columns if col in llm_df.columns]
    return llm_df[available_columns].sort_values(by=["start_time"], ascending=True, na_position="last").reset_index(drop=True)


def build_retrieval_dataframe_from_phoenix(spans_df: pd.DataFrame) -> pd.DataFrame:
    """Explode Phoenix retriever spans into one row per retrieved document.

    Purpose:
    - Phoenix retriever spans can contain an `attributes.retrieval.documents` list.
    - This helper normalizes those nested document records into a flat evaluation table.

    Output:
    - DataFrame with one row per retrieved document.
    """
    retrieval_column = "attributes.retrieval.documents"
    if spans_df.empty or retrieval_column not in spans_df.columns:
        return pd.DataFrame()

    retrieval_spans = spans_df[spans_df[retrieval_column].notna()].copy()
    rows: list[dict[str, Any]] = []

    for _, span_row in retrieval_spans.iterrows():
        documents = span_row.get(retrieval_column) or []
        if not isinstance(documents, list):
            continue

        for rank, document in enumerate(documents, start=1):
            if not isinstance(document, dict):
                continue

            rows.append(
                {
                    "trace_id": span_row.get("context.trace_id"),
                    "span_id": span_row.get("context.span_id"),
                    "retrieval_span_name": span_row.get("name"),
                    "session_id": span_row.get("attributes.session.id"),
                    "query_text": span_row.get("attributes.input.value"),
                    "document_rank": rank,
                    "document_score": document.get("document.score"),
                    "document_content_preview": (document.get("document.content") or "")[:500],
                    "document_raw": _safe_json_text(document),
                }
            )

    retrieval_df = pd.DataFrame(rows)
    if not retrieval_df.empty:
        retrieval_df = retrieval_df.sort_values(by=["trace_id", "document_rank"], ascending=[True, True], na_position="last")
    return retrieval_df.reset_index(drop=True)


phoenix_client = create_phoenix_client(EVAL_CONFIG)
catalog_df = fetch_trace_catalog_dataframe(
    client=phoenix_client,
    project_name=EVAL_CONFIG.phoenix_project_name,
    limit=EVAL_CONFIG.phoenix_trace_limit,
)
all_project_spans_df = fetch_spans_dataframe(
    client=phoenix_client,
    project_name=EVAL_CONFIG.phoenix_project_name,
    limit=EVAL_CONFIG.phoenix_span_limit,
)
available_session_ids = fetch_available_session_ids(all_project_spans_df)

print(f"Phoenix endpoint: {EVAL_CONFIG.phoenix_base_url}")
print(f"Phoenix project:  {EVAL_CONFIG.phoenix_project_name}")
print(f"Install command if Phoenix packages are missing: {PHOENIX_INSTALL_COMMAND}")
print(f"Detected traces in catalog: {len(catalog_df)}")
print(f"Available Phoenix session ids: {available_session_ids[:10]}")
catalog_df.head(20)

c:\Users\jason\miniconda3\envs\ai\Lib\site-packages\authlib\_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


Phoenix endpoint: http://localhost:6006
Phoenix project:  rag-news-chatbot
Install command if Phoenix packages are missing: pip install arize-phoenix arize-phoenix-otel openinference-instrumentation-llama-index
Detected traces in catalog: 31
Available Phoenix session ids: ['Donald_Trump', 'Test_Chat_session', 'chat_20260701_223412', 'chat_20260701_230551', 'chat_20260701_231246']


,trace_row_id,trace_id,project_id,start_time,end_time,latency_ms,token_count_prompt,token_count_completion,token_count_total
0,VHJhY2U6NjI=,6b23922c698c812d440d85b8c5d9e766,UHJvamVjdDo0,2026-07-10 05:28:45.010993+00:00,2026-07-10 05:29:56.126233+00:00,71115.240,7503,300,7803
1,VHJhY2U6NTY=,25e1fa6a8dd8f9d7b8cd65e777aaf6da,UHJvamVjdDo0,2026-07-06 06:46:35.894681+00:00,2026-07-06 06:47:47.799154+00:00,71904.473,7017,284,7301
2,VHJhY2U6NTU=,3d38d313b4cfaa5abb911e2c051850a4,UHJvamVjdDo0,2026-07-06 06:44:55.329681+00:00,2026-07-06 06:46:06.377400+00:00,71047.719,6887,121,7008
3,VHJhY2U6NTQ=,a675b3a81576d7d789aeed587f4b217e,UHJvamVjdDo0,2026-07-06 06:30:20.036784+00:00,2026-07-06 06:31:20.962444+00:00,60925.660,5533,191,5724
4,VHJhY2U6NTM=,aa761f0791e762e4a6696c1b7734632f,UHJvamVjdDo0,2026-07-06 06:23:06.454026+00:00,2026-07-06 06:24:03.668721+00:00,57214.695,4676,127,4803
5,VHJhY2U6NDc=,3fef4a593df9d75ab1876de6a7f83e26,UHJvamVjdDo0,2026-07-06 03:13:45.062124+00:00,2026-07-06 03:15:00.861398+00:00,75799.274,4062,339,4401
6,VHJhY2U6NDY=,a4d0a915da73b872bea0f403480ee9c7,UHJvamVjdDo0,2026-07-06 03:11:20.006984+00:00,2026-07-06 03:11:59.023908+00:00,39016.924,2581,248,2829
7,VHJhY2U6NDA=,82b56138ab46c2ba0b0a1d516f10bb39,UHJvamVjdDo0,2026-07-01 15:12:46.078460+00:00,2026-07-01 15:13:46.047206+00:00,59968.746,3950,30,3980
8,VHJhY2U6Mzk=,f6643cf69502338c2f8c768a15d1e4c8,UHJvamVjdDo0,2026-07-01 15:09:10.424399+00:00,2026-07-01 15:10:30.676882+00:00,80252.483,4688,368,5056
9,VHJhY2U6Mzg=,d86fb67cd624e2fec720109f9af4cdfd,UHJvamVjdDo0,2026-07-01 15:07:30.133315+00:00,2026-07-01 15:08:28.780206+00:00,58646.891,3659,220,3879


In [3]:
# --- Example usage: pick one Phoenix session id, then run this cell ---

# For this project, the Phoenix session id usually matches the saved chat id used during inference.
TARGET_SESSION_ID = available_session_ids[0] if "available_session_ids" in globals() and available_session_ids else None


def get_phoenix_evaluation_bundle(
    session_id: str,
    config: EvaluationConfig,
 ) -> dict[str, pd.DataFrame]:
    """Build the full evaluation bundle directly from Phoenix serve.

    Purpose:
    - Use Phoenix as the single source of evaluation truth for the selected session.
    - Return multiple dataframes because Phoenix data exists at several granularities: traces, spans,
      turn spans, LLM spans, and retrieved documents.

    Output:
    - Dictionary containing the following dataframes:
      `trace_df`, `session_spans_df`, `span_overview_df`, `turn_df`, `llm_df`, and `retrieval_df`.
    """
    client = create_phoenix_client(config)

    trace_df = fetch_trace_catalog_dataframe(
        client=client,
        project_name=config.phoenix_project_name,
        limit=config.phoenix_trace_limit,
        session_id=session_id,
    )

    all_spans_df = fetch_spans_dataframe(
        client=client,
        project_name=config.phoenix_project_name,
        limit=config.phoenix_span_limit,
    )

    session_spans_df = filter_spans_by_session(all_spans_df, session_id=session_id)
    span_overview_df = build_span_overview_dataframe(session_spans_df)
    turn_df = build_turn_dataframe_from_phoenix(session_spans_df)
    llm_df = build_llm_dataframe_from_phoenix(session_spans_df)
    retrieval_df = build_retrieval_dataframe_from_phoenix(session_spans_df)

    return {
        "trace_df": trace_df,
        "session_spans_df": session_spans_df,
        "span_overview_df": span_overview_df,
        "turn_df": turn_df,
        "llm_df": llm_df,
        "retrieval_df": retrieval_df,
    }


def save_phoenix_evaluation_outputs(
    session_id: str,
    project_root: Path,
    phoenix_bundle: dict[str, pd.DataFrame],
 ) -> Path:
    """Persist Phoenix evaluation dataframes as CSV files.

    Purpose:
    - Export notebook evaluation artifacts into files that can be audited or shared.

    Output:
    - Path to the output directory containing the generated CSV files.
    """
    output_dir = project_root / "result" / "evaluation_outputs"
    output_dir.mkdir(parents=True, exist_ok=True)

    for dataframe_name, dataframe in phoenix_bundle.items():
        dataframe.to_csv(
            output_dir / f"{session_id}_{dataframe_name}.csv",
            index=False,
            encoding="utf-8",
        )

    return output_dir


if not TARGET_SESSION_ID:
    print("No Phoenix session ids were found. Run traced inference first, then rerun Cell 3.")
else:
    phoenix_bundle = get_phoenix_evaluation_bundle(
        session_id=TARGET_SESSION_ID,
        config=EVAL_CONFIG,
    )

    trace_df = phoenix_bundle["trace_df"]
    session_spans_df = phoenix_bundle["session_spans_df"]
    span_overview_df = phoenix_bundle["span_overview_df"]
    turn_df = phoenix_bundle["turn_df"]
    llm_df = phoenix_bundle["llm_df"]
    retrieval_df = phoenix_bundle["retrieval_df"]

    print("=== Phoenix Evaluation Context ===")
    print(f"Phoenix endpoint: {EVAL_CONFIG.phoenix_base_url}")
    print(f"Phoenix project:  {EVAL_CONFIG.phoenix_project_name}")
    print(f"Session id:       {TARGET_SESSION_ID}")
    print(f"Trace rows:       {len(trace_df)}")
    print(f"Span rows:        {len(session_spans_df)}")
    print(f"Turn rows:        {len(turn_df)}")
    print(f"LLM rows:         {len(llm_df)}")
    print(f"Retrieval rows:   {len(retrieval_df)}")

    print("\n=== Trace DataFrame ===")
    display(trace_df.head(20))

    print("\n=== Span Overview DataFrame ===")
    display(span_overview_df.head(20))

    print("\n=== Turn DataFrame ===")
    display(turn_df.head(20))

    print("\n=== LLM DataFrame ===")
    display(llm_df.head(20))

    print("\n=== Retrieval DataFrame ===")
    display(retrieval_df.head(30))

    output_dir = save_phoenix_evaluation_outputs(
        session_id=TARGET_SESSION_ID,
        project_root=PROJECT_ROOT,
        phoenix_bundle=phoenix_bundle,
    )
    print(f"Saved Phoenix evaluation CSV files to: {output_dir}")

=== Phoenix Evaluation Context ===
Phoenix endpoint: http://localhost:6006
Phoenix project:  rag-news-chatbot
Session id:       Donald_Trump
Trace rows:       12
Span rows:        218
Turn rows:        16
LLM rows:         44
Retrieval rows:   275

=== Trace DataFrame ===


,trace_row_id,trace_id,project_id,start_time,end_time,latency_ms,token_count_prompt,token_count_completion,token_count_total
0,VHJhY2U6MzQ=,f0aa2aa02b3f97a15b0fa0df2828d40c,UHJvamVjdDo0,2026-07-01 14:30:38.183347+00:00,2026-07-01 14:33:26.190100+00:00,168006.753,13093,590,13683
1,VHJhY2U6MzM=,d22ca5fecd6107dadbf7f3efc18f9360,UHJvamVjdDo0,2026-07-01 11:54:30.660439+00:00,2026-07-01 11:56:34.059492+00:00,123399.053,12439,833,13272
2,VHJhY2U6MzI=,f3d9a2e5aa72fedc29139ec8db64f23a,UHJvamVjdDo0,2026-07-01 10:58:06.481094+00:00,2026-07-01 11:00:13.791548+00:00,127310.454,12365,666,13031
3,VHJhY2U6MzA=,f3ea842d883a474fd5e8111c901d7634,UHJvamVjdDo0,2026-07-01 10:49:18.945317+00:00,2026-07-01 10:50:56.255361+00:00,97310.044,11971,503,12474
4,VHJhY2U6Mjg=,4b62ed0817cde49391b68aaa0f5d0d42,UHJvamVjdDo0,2026-07-01 08:58:47.214192+00:00,2026-07-01 09:00:43.890761+00:00,116676.569,12309,604,12913
5,VHJhY2U6MTc=,9b5395c6ccf40bd0405defcf737ab0f0,UHJvamVjdDo0,2026-07-01 04:23:23.881876+00:00,2026-07-01 04:24:50.978162+00:00,87096.286,9341,617,9958
6,VHJhY2U6MTY=,a27cc11ec7d5d43baf7292a1a7fd19a2,UHJvamVjdDo0,2026-07-01 04:19:01.846737+00:00,2026-07-01 04:20:18.228971+00:00,76382.234,7688,499,8187
7,VHJhY2U6MTU=,6fbae27a86ec3f22f0c08834fcdbb38b,UHJvamVjdDo0,2026-07-01 04:16:39.900245+00:00,2026-07-01 04:17:57.592931+00:00,77692.686,4345,408,4753
8,VHJhY2U6MTQ=,fbb1d06fc54a882ed06afdf2986d9f4d,UHJvamVjdDo0,2026-06-30 17:42:52.331695+00:00,2026-06-30 17:43:49.239148+00:00,56907.453,5571,384,5955
9,VHJhY2U6MTM=,80eb72ab82e62354bc7969f3776c6b51,UHJvamVjdDo0,2026-06-30 17:21:58.744191+00:00,2026-06-30 17:22:48.801666+00:00,50057.475,5128,363,5491



=== Span Overview DataFrame ===


,name,span_kind,status_code,start_time,end_time,context.trace_id,context.span_id,parent_id,attributes.session.id,interface,attributes.llm.model_name,attributes.llm.token_count.prompt,attributes.llm.token_count.completion,attributes.llm.token_count.total,attributes.input.value,attributes.output.value
0,Ollama.chat,LLM,OK,2026-07-01 14:31:36.001830+00:00,2026-07-01 14:33:24.106008+00:00,f0aa2aa02b3f97a15b0fa0df2828d40c,dc53047d4d707bc0,f12554dba6b418f0,Donald_Trump,chainlit,gemma3:1b,6587.0,526.0,7113.0,"{""messages"": [""ChatMessage(role=<MessageRole.S...",assistant: You’ve articulated a very important...
1,Ollama.predict,LLM,OK,2026-07-01 14:31:35.998692+00:00,2026-07-01 14:33:26.154738+00:00,f0aa2aa02b3f97a15b0fa0df2828d40c,f12554dba6b418f0,b4dbb817a1788462,Donald_Trump,chainlit,gemma3:1b,NaN,NaN,NaN,"{""prompt"": ""ChatPromptTemplate(metadata={'prom...",You’ve articulated a very important and nuance...
2,DefaultRefineProgram.__call__,CHAIN,OK,2026-07-01 14:31:35.997571+00:00,2026-07-01 14:33:26.160993+00:00,f0aa2aa02b3f97a15b0fa0df2828d40c,b4dbb817a1788462,d4b48245fdaa0fda,Donald_Trump,chainlit,NaN,NaN,NaN,NaN,"{""kwds"": {""context_str"": ""source_folder: hk_fr...","{""query_satisfied"":true,""answer"":""You’ve artic..."
3,TokenTextSplitter.split_text,CHAIN,OK,2026-07-01 14:31:35.985524+00:00,2026-07-01 14:31:35.990348+00:00,f0aa2aa02b3f97a15b0fa0df2828d40c,271c4f74dd721619,d4b48245fdaa0fda,Donald_Trump,chainlit,NaN,NaN,NaN,NaN,"{""text"": ""source_folder: hk_free_press_news\nf...","[""source_folder: hk_free_press_news\nfile_path..."
4,CompactAndRefine.get_response,CHAIN,OK,2026-07-01 14:31:35.978265+00:00,2026-07-01 14:33:26.165850+00:00,f0aa2aa02b3f97a15b0fa0df2828d40c,d4b48245fdaa0fda,5504286d04446040,Donald_Trump,chainlit,NaN,NaN,NaN,NaN,Trump rather than Bannon hate Joe Biden,You’ve articulated a very important and nuance...
5,TokenTextSplitter.split_text,CHAIN,OK,2026-07-01 14:31:35.963101+00:00,2026-07-01 14:31:35.970976+00:00,f0aa2aa02b3f97a15b0fa0df2828d40c,4f76451717aa68de,5504286d04446040,Donald_Trump,chainlit,NaN,NaN,NaN,NaN,"{""text"": ""source_folder: hk_free_press_news\nf...","[""source_folder: hk_free_press_news\nfile_path..."
6,CompactAndRefine.get_response,CHAIN,OK,2026-07-01 14:31:35.949847+00:00,2026-07-01 14:33:26.170729+00:00,f0aa2aa02b3f97a15b0fa0df2828d40c,5504286d04446040,cc4029197a9e64c7,Donald_Trump,chainlit,NaN,NaN,NaN,NaN,"{""query_str"": ""Trump rather than Bannon hate J...",You’ve articulated a very important and nuance...
7,CompactAndRefine.synthesize,CHAIN,OK,2026-07-01 14:31:35.948512+00:00,2026-07-01 14:33:26.175640+00:00,f0aa2aa02b3f97a15b0fa0df2828d40c,cc4029197a9e64c7,7d53ab57a4218d82,Donald_Trump,chainlit,NaN,NaN,NaN,NaN,Trump rather than Bannon hate Joe Biden,You’ve articulated a very important and nuance...
8,BM25Retriever._retrieve,RETRIEVER,OK,2026-07-01 14:31:35.905769+00:00,2026-07-01 14:31:35.914591+00:00,f0aa2aa02b3f97a15b0fa0df2828d40c,64a2fd71e019e610,7666e5855d17a3a1,Donald_Trump,chainlit,NaN,NaN,NaN,NaN,"{""query_bundle"": {""query_str"": ""**Why did Step...","[""<NodeWithScore(node=TextNode(id_=fb9a6140-2c..."
9,BM25Retriever.retrieve,RETRIEVER,OK,2026-07-01 14:31:35.900393+00:00,2026-07-01 14:31:35.923556+00:00,f0aa2aa02b3f97a15b0fa0df2828d40c,7666e5855d17a3a1,564d8a94509ce54f,Donald_Trump,chainlit,NaN,NaN,NaN,NaN,**Why did Stephen Bannon consistently criticiz...,"[""<NodeWithScore(node=TextNode(id_=fb9a6140-2c..."



=== Turn DataFrame ===


...


=== LLM DataFrame ===


,name,start_time,end_time,status_code,context.trace_id,context.span_id,attributes.session.id,interface,attributes.llm.model_name,attributes.llm.input_messages,attributes.llm.output_messages,attributes.llm.invocation_parameters,attributes.llm.token_count.prompt,attributes.llm.token_count.completion,attributes.llm.token_count.total,attributes.output.value
0,Ollama.predict,2026-06-30 17:20:42.494305+00:00,2026-06-30 17:21:18.445003+00:00,OK,7154de4aac75a7442dd6b378a732b535,e756439b203a8b90,Donald_Trump,notebook,gemma3:1b,None,None,"{""context_window"":32768,""num_output"":256,""is_c...",NaN,NaN,NaN,Donald Trump is an American businessman and po...
1,Ollama.chat,2026-06-30 17:20:42.496050+00:00,2026-06-30 17:21:18.444573+00:00,OK,7154de4aac75a7442dd6b378a732b535,83bef5bb95fb03d4,Donald_Trump,notebook,gemma3:1b,[{'message.content': ' The following is a fr...,[{'message.content': 'Donald Trump is an Ameri...,"{""context_window"":32768,""num_output"":256,""is_c...",3258.0,182.0,3440.0,assistant: Donald Trump is an American busines...
2,Ollama.complete,2026-06-30 17:21:18.491248+00:00,2026-06-30 17:21:21.069975+00:00,OK,12e34b0e14146ab58e9959ba8705c111,37596af57b6491f4,Donald_Trump,notebook,gemma3:1b,None,None,"{""context_window"":32768,""num_output"":256,""is_c...",259.0,9.0,268.0,Did Donald Trump meet with Xi Jinping?
3,Ollama.chat,2026-06-30 17:21:18.493967+00:00,2026-06-30 17:21:21.069446+00:00,OK,12e34b0e14146ab58e9959ba8705c111,cdd3799bb4452909,Donald_Trump,notebook,gemma3:1b,[{'message.content': ' Given the following c...,[{'message.content': 'Did Donald Trump meet wi...,"{""context_window"":32768,""num_output"":256,""is_c...",259.0,9.0,268.0,assistant: Did Donald Trump meet with Xi Jinping?
4,Ollama.predict,2026-06-30 17:21:21.218562+00:00,2026-06-30 17:21:58.702976+00:00,OK,12e34b0e14146ab58e9959ba8705c111,21c9b5d54f8616ce,Donald_Trump,notebook,gemma3:1b,None,None,"{""context_window"":32768,""num_output"":256,""is_c...",NaN,NaN,NaN,"Yes, Donald Trump did meet with Xi Jinping, th..."
5,Ollama.chat,2026-06-30 17:21:21.221354+00:00,2026-06-30 17:21:58.702604+00:00,OK,12e34b0e14146ab58e9959ba8705c111,d06ff0f37c527e21,Donald_Trump,notebook,gemma3:1b,[{'message.content': ' The following is a fr...,"[{'message.content': 'Yes, Donald Trump did me...","{""context_window"":32768,""num_output"":256,""is_c...",4225.0,70.0,4295.0,"assistant: Yes, Donald Trump did meet with Xi ..."
6,Ollama.complete,2026-06-30 17:21:58.745674+00:00,2026-06-30 17:22:02.038215+00:00,OK,80eb72ab82e62354bc7969f3776c6b51,f24dfb8fceae0636,Donald_Trump,notebook,gemma3:1b,None,None,"{""context_window"":32768,""num_output"":256,""is_c...",348.0,15.0,363.0,What is the relationship between Donald Trump ...
7,Ollama.chat,2026-06-30 17:21:58.747722+00:00,2026-06-30 17:22:02.037605+00:00,OK,80eb72ab82e62354bc7969f3776c6b51,83a8782054f0cd5c,Donald_Trump,notebook,gemma3:1b,[{'message.content': ' Given the following c...,[{'message.content': 'What is the relationship...,"{""context_window"":32768,""num_output"":256,""is_c...",348.0,15.0,363.0,assistant: What is the relationship between Do...
8,Ollama.predict,2026-06-30 17:22:02.249596+00:00,2026-06-30 17:22:48.797332+00:00,OK,80eb72ab82e62354bc7969f3776c6b51,2246b9637593ed0e,Donald_Trump,notebook,gemma3:1b,None,None,"{""context_window"":32768,""num_output"":256,""is_c...",NaN,NaN,NaN,The relationship between Donald Trump and Step...
9,Ollama.chat,2026-06-30 17:22:02.252398+00:00,2026-06-30 17:22:48.796941+00:00,OK,80eb72ab82e62354bc7969f3776c6b51,c678095ab1cf9b9e,Donald_Trump,notebook,gemma3:1b,[{'message.content': ' The following is a fr...,[{'message.content': 'The relationship between...,"{""context_window"":32768,""num_output"":256,""is_c...",4432.0,333.0,4765.0,assistant: The relationship between Donald Tru...



=== Retrieval DataFrame ===


,trace_id,span_id,retrieval_span_name,session_id,query_text,document_rank,document_score,document_content_preview,document_raw
0,12e34b0e14146ab58e9959ba8705c111,679e33520de2061d,BM25Retriever.retrieve,Donald_Trump,Did Donald Trump meet with Xi Jinping?,1,9.945602,President Donald Trump said he had made “fanta...,"{""document.score"": 9.945602417, ""document.cont..."
1,12e34b0e14146ab58e9959ba8705c111,bcf46aa47258fcc9,VectorIndexRetriever.retrieve,Donald_Trump,Did Donald Trump meet with Xi Jinping?,1,0.777383,US President Donald Trump said Wednesday he wi...,"{""document.score"": 0.7773828492, ""document.con..."
2,12e34b0e14146ab58e9959ba8705c111,7879ce79e3f32aba,HybridRetriever.retrieve,Donald_Trump,Did Donald Trump meet with Xi Jinping?,1,0.032002,Delegations from China and the United States m...,"{""document.score"": 0.0320020481, ""document.con..."
3,12e34b0e14146ab58e9959ba8705c111,679e33520de2061d,BM25Retriever.retrieve,Donald_Trump,Did Donald Trump meet with Xi Jinping?,2,9.771670,Delegations from China and the United States m...,"{""document.score"": 9.7716703415, ""document.con..."
4,12e34b0e14146ab58e9959ba8705c111,bcf46aa47258fcc9,VectorIndexRetriever.retrieve,Donald_Trump,Did Donald Trump meet with Xi Jinping?,2,0.760676,By Danny Kemp\n\nChinese President Xi Jinping ...,"{""document.score"": 0.7606758404, ""document.con..."
5,12e34b0e14146ab58e9959ba8705c111,7879ce79e3f32aba,HybridRetriever.retrieve,Donald_Trump,Did Donald Trump meet with Xi Jinping?,2,0.031778,US President Donald Trump said Wednesday he wi...,"{""document.score"": 0.031778058000000005, ""docu..."
6,12e34b0e14146ab58e9959ba8705c111,679e33520de2061d,BM25Retriever.retrieve,Donald_Trump,Did Donald Trump meet with Xi Jinping?,3,9.671169,Beijing is also likely to use Trump’s weakened...,"{""document.score"": 9.671169281000001, ""documen..."
7,12e34b0e14146ab58e9959ba8705c111,bcf46aa47258fcc9,VectorIndexRetriever.retrieve,Donald_Trump,Did Donald Trump meet with Xi Jinping?,3,0.758652,Delegations from China and the United States m...,"{""document.score"": 0.7586517352000001, ""docume..."
8,12e34b0e14146ab58e9959ba8705c111,7879ce79e3f32aba,HybridRetriever.retrieve,Donald_Trump,Did Donald Trump meet with Xi Jinping?,3,0.031778,President Donald Trump said he had made “fanta...,"{""document.score"": 0.031778058000000005, ""docu..."
9,12e34b0e14146ab58e9959ba8705c111,679e33520de2061d,BM25Retriever.retrieve,Donald_Trump,Did Donald Trump meet with Xi Jinping?,4,9.646190,US President Donald Trump heads for a superpow...,"{""document.score"": 9.6461896896, ""document.con..."


Saved Phoenix evaluation CSV files to: C:\Program Files\Studying\coding\RAG_project\result\evaluation_outputs


In [4]:
# Quick inspection helper for interactive notebook work.
# After running the main Phoenix evaluation cell, this cell shows the available dataframe names.
list(phoenix_bundle.keys()) if "phoenix_bundle" in globals() else "Run Cell 4 first."

['trace_df',
 'session_spans_df',
 'span_overview_df',
 'turn_df',
 'llm_df',
 'retrieval_df']